# Semantic Segmentation & GVI Computation

Runs Mask2Former (ADE20K) on summer street-view images to compute:
- Per-class pixel ratios for all 150 ADE20K classes
- Summer GVI = sum of six vegetation class ratios (tree, grass, plant, palm, flower, field)

Output CSV matches the input format expected by `03_inference_batch.ipynb`.

> Model: `facebook/mask2former-swin-large-ade-semantic`


In [ ]:
%pip install -q transformers torch torchvision Pillow numpy pandas

## Step 0 — Configuration


In [ ]:
IMAGE_DIR  = "../data/sample/images"   # Folder containing summer street-view images
POINTS_CSV = ""                      
 

# Used only when POINTS_CSV is blank
CITY    = "Seoul"
COUNTRY = "South Korea"

OUTPUT_DIR = "../results"
OUTPUT_CSV = "segmentation_results.csv"  # Input CSV for 03_inference_batch.ipynb

SAVE_MASKS = False   

## Step 1 — Load Model

In [ ]:
import torch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

MODEL_ID = "facebook/mask2former-swin-large-ade-semantic"
processor  = AutoImageProcessor.from_pretrained(
    MODEL_ID, size={"height": 640, "width": 640}
)
seg_model  = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_ID)
seg_model  = seg_model.to(device).eval()
print("Model loaded.")

## Step 2 — Setup Labels & Helpers

In [ ]:
import re, os, glob, gc, csv, json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

# ── ADE20K label maps ───────────────────────────────────────────────────────
id2label = {int(k): v for k, v in seg_model.config.id2label.items()}
num_labels = seg_model.config.num_labels

# Six vegetation classes used for GVI (paper Section 3.3)
VEG_CLASSES = ['tree', 'grass', 'palm', 'plant', 'flower', 'field']
veg_class_ids = [
    cid for cid, label in id2label.items()
    if label.lower() in VEG_CLASSES
]
veg_id2name = {cid: id2label[cid].lower() for cid in veg_class_ids}
print(f"Vegetation class IDs: {veg_id2name}")


def sanitize(label: str) -> str:
    s = re.sub(r'\s+', '_', str(label).lower().strip())
    return re.sub(r'[^a-z0-9_]', '', s) or 'unknown'


# Column names for all 150 ADE20K classes
all_class_cols = [f"class_{i}_{sanitize(id2label[i])}" for i in range(num_labels)]

# Vegetation-only output columns (matches metadata_sample.csv)
VEG_COL_MAP = {cid: f"class_{cid}_{veg_id2name[cid]}" for cid in veg_class_ids}
print(f"Output veg columns: {list(VEG_COL_MAP.values())}")


# ── Segmentation helpers ────────────────────────────────────────────────────
def run_segmentation(image_pil: Image.Image) -> np.ndarray:
    """Run Mask2Former; returns 640x640 class label map."""
    encoding = processor(image_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = seg_model(**encoding)
    seg = processor.post_process_semantic_segmentation(
        outputs, target_sizes=[(640, 640)]
    )[0]
    return seg.cpu().numpy()


def per_class_ratios(seg_np: np.ndarray) -> np.ndarray:
    """Pixel ratio per ADE20K class (float32)."""
    total = seg_np.size
    counts = np.bincount(seg_np.ravel(), minlength=num_labels).astype(np.float32)
    return counts / (total + 1e-12)


def make_veg_mask(seg_np: np.ndarray, save_path: str):
    """Save multi-class vegetation mask PNG (value 0 = non-veg, 1..N per class)."""
    mask = np.zeros(seg_np.shape, dtype=np.uint8)
    for save_val, cid in enumerate(veg_class_ids, start=1):
        mask[seg_np == cid] = save_val
    Image.fromarray(mask, mode='L').save(save_path)

## Step 3 — Run Segmentation

Processes every image in `IMAGE_DIR`, writing results row-by-row.


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Recursive glob: 01_gsv_collection saves images under images/{city}/{pano_id}/
image_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, '**', ext), recursive=True))
image_paths = sorted(image_paths)
if not image_paths:
    raise FileNotFoundError(f"No images found in: {IMAGE_DIR}")
print(f"Found {len(image_paths)} images")

csv_path  = OUTPUT_PATH / "_seg_raw.csv"           # intermediate: all 150 class ratios
mask_dir  = OUTPUT_PATH / "vegetation_masks"
if SAVE_MASKS:
    mask_dir.mkdir(parents=True, exist_ok=True)

columns = ['image_id'] + all_class_cols
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()

    for idx, img_path in enumerate(image_paths):
        img_name  = Path(img_path).stem
        image_pil = Image.open(img_path).convert('RGB')
        print(f"[{idx+1}/{len(image_paths)}] {img_name}")

        seg_np  = run_segmentation(image_pil)
        ratios  = per_class_ratios(seg_np)

        row = {'image_id': img_name}
        row.update({all_class_cols[i]: float(ratios[i]) for i in range(num_labels)})
        writer.writerow(row)

        if SAVE_MASKS:
            make_veg_mask(seg_np, str(mask_dir / f"{img_name}_vegmask.png"))

        del image_pil, seg_np, ratios
        if idx % 25 == 0:
            gc.collect()

print(f"Raw segmentation saved -> {csv_path}")


## Step 4 — Compute GVI & Build Output CSV

GVI = sum of pixel ratios for the six vegetation classes (paper Eq. 1, Section 3.3).  
If `POINTS_CSV` is provided, merges latitude/longitude/city/country from the points metadata.

In [ ]:
import re

df_seg = pd.read_csv(csv_path)

# ── Compute summer GVI and extract vegetation class ratios ──────────────
for cid, col_name in VEG_COL_MAP.items():
    df_seg[col_name] = df_seg[all_class_cols[cid]]

veg_output_cols = list(VEG_COL_MAP.values())
df_seg['GVI'] = df_seg[veg_output_cols].sum(axis=1)

# ── Derive point_id (= pano_id) + heading from 'summer_{pano_id}_{heading}' ──
# pano_id may contain '_' / '-', so anchor the regex on the known heading values.
def parse_stem(stem):
    m = re.match(r'^summer_(.+)_(0|90|180|270)$', str(stem))
    return (m.group(1), int(m.group(2))) if m else (str(stem), 0)

df_seg[['point_id', 'heading']] = df_seg['image_id'].apply(
    lambda s: pd.Series(parse_stem(s)))

df_seg['filepath'] = df_seg['image_id'].apply(
    lambda stem: str(next(Path(IMAGE_DIR).glob(f"**/{stem}.*"), stem))
)

# ── Merge lat/lon/city/country from points metadata ──────────────
if POINTS_CSV:
    df_pts = pd.read_csv(POINTS_CSV)
    merge_key = 'point_id' if 'point_id' in df_pts.columns else 'image_id'
    df_seg = df_seg.merge(df_pts[[merge_key, 'lat', 'lon', 'city', 'country']],
                          left_on='point_id', right_on=merge_key, how='left')
else:
    df_seg['lat']     = None
    df_seg['lon']     = None
    df_seg['city']    = CITY
    df_seg['country'] = COUNTRY

# One row per image (up to 4 per point)
out_cols = ['image_id', 'point_id', 'heading', 'filepath',
            'lat', 'lon', 'city', 'country', 'GVI'] + veg_output_cols
df_out = df_seg[out_cols]

out_path = OUTPUT_PATH / OUTPUT_CSV
df_out.to_csv(out_path, index=False, encoding='utf-8')
print(f"Output saved -> {out_path}")
print(f"  Rows (images): {len(df_out)}  |  Points: {df_out['point_id'].nunique()}")
print(f"  GVI range: {df_out['GVI'].min():.3f} – {df_out['GVI'].max():.3f}")
df_out.head(3)


## Step 5 — Save Vegetation Mask Value Map (optional)

In [ ]:
mask_value_map = {"0": "non_vegetation"}
for save_val, cid in enumerate(veg_class_ids, start=1):
    mask_value_map[str(save_val)] = veg_id2name[cid]

map_path = OUTPUT_PATH / "vegetation_mask_value_map.json"
with open(map_path, 'w', encoding='utf-8') as f:
    json.dump(mask_value_map, f, ensure_ascii=False, indent=4)
print(f"Mask value map saved -> {map_path}")
print(mask_value_map)